# Classification with PyTorch/TensorFlow

Lab Assignment from [AI for Beginners Curriculum](https://github.com/microsoft/ai-for-beginners).

## Part 1: Iris Classification

Iris Dataset contains 150 records of 3 different classes of irises. Each record contains 4 numeric parameters: sepal length/width and petal length/width. It is an example of a simple dataset, for which you do not need a powerful neural network.

### Getting the Dataset

Iris dataset is build into Scikit Learn, so we can easily get it:

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
features = iris['data']
labels = iris['target']
class_names = iris['target_names']
feature_names = iris['feature_names']

print(f"Features: {feature_names}, Classes: {class_names}")

### Visualize the Data

In many cases, it makes sense to visualize the data to see if they look separable - it would assure us that we should be able to build good classification model. Because we have a few features, we can build a series of pairwise 2D scatter plots, showing different classes by different dot colors. This can be automatically done by a package called **seaborn**:

In [ ]:
import seaborn as sns
import pandas as pd

df = pd.DataFrame(features,columns=feature_names).join(pd.DataFrame(labels,columns=['Label']))

df

In [ ]:
sns.pairplot(df, hue='Label')

### Normalize and Encode the Data

To prepare data to neural network training, we need to normalize inputs in the range [0..1]. This can be done either using plain `numpy` operations, or [Scikit Learn methods](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.normalize.html).

Also, you need to decide if you want target label to be one-hot encoded or not. PyTorch and TensorFlow allow you feed in class number either as an integer (from 0 to N-1), or as one-hot encoded vector. When creating neural network structure, you need to specify loss function accordingly (eg. *sparse categorical crossentropy* for numeric representation, and *crossentropy loss* for one-hot encoding). One-hot encoding can also be [done using Sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html), or by using this piece of code:

```python
n_values = np.max(labels) + 1
labels_onehot = np.eye(n_values)[labels]
``` 

In [ ]:
# Split the data
from sklearn.model_selection import train_test_split

X = df[feature_names].values
y = df['Label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Code to normalize and encode the data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

X_train_scale

### Split the Data into Train and Test

Since we do not have separate train and test dataset, we need to split it intro train and test dataset [using Sklearn](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)

In [ ]:

X_test_scale

### Define and Train Neural Network

Now you are ready to go, import your preferred framework, define the neural network and start training, observing the behavior of train and validation accuracy.

In [ ]:
# Define the network
import torch

torch.__version__

import torch.nn as nn
import torch.functional as F


class NeuralNetworkLayer1(nn.Module):
    def __init__(self):
        super().__init__()  # super(NeuralNetworkLayer1, self).__init__()

        self.fc1 = nn.Linear(4, 3)

    def forward(self, x):
        x = self.fc1(x)
        return x


class NeuralNetworkLayer1ReLu(nn.Module):
    def __init__(self):
        super().__init__()  # super(NeuralNetworkLayer1ReLu,self).__init__

        self.fc1 = nn.Linear(4, 3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        return x


class NeuralNetworkLayer2(nn.Module):
    def __init__(self, hidden_size):
        # 8 for small hidden size
        # 32 for medium hidden size
        # 128 for large hidden size
        super().__init__()  # super(NeuralNetworkLayer,self).__init__()

        self.fc1 = nn.Linear(4, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


class NeuralNetworkLayer3(nn.Module):
    def __init__(
        self, first_layer_hidden_size, second_layer_hidden_size, *args, **kwargs
    ) -> None:
        super().__init__(*args, **kwargs)
        self.fc1 = nn.Linear(4, first_layer_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(first_layer_hidden_size, second_layer_hidden_size)
        self.fc3 = nn.Linear(second_layer_hidden_size, 3)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x


class UniversalNet(nn.Module):
    def __init__(self, layer, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)

        self.layers = nn.ModuleList()
        for i in range(len(layer) - 1):
            self.layers.append(nn.Linear(layer[i], layer[i + 1]))
        self.relu = nn.ReLU()

    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = self.relu(self.layers[i](x))

        x = self.layers[-1](x)

        return x

In [ ]:
# Train the network
def train_model(model, X_train, y_train, epochs=200, lr=0.01):
    # define the loss function(sparse)
    criterion = nn.CrossEntropyLoss()

    # define optimizer
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    # train loop
    for epoch in range(epochs):
        # Clear the grad from the last round
        optimizer.zero_grad()

        # Forward propagation
        outputs = model.forward(X_train)

        # Calculate the loss
        loss = criterion(outputs, y_train)
        
        # Backward propagation
        loss.backward()
        
        # Update the weight
        optimizer.step()
        
        print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}')

    return model

In [ ]:
# Visualize train/validation accuracy graph
X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).view(-1)

X_train_tensor.shape

In [ ]:
y_train_tensor.shape

### Layer 2

In [ ]:
model = UniversalNet([4, 8, 3])

In [ ]:
trained_model = train_model(model, X_train_tensor, y_train_tensor)

### Experiment

Now you can experiment with different network architectures to see how it affects the result. Try:
1. One-layer network with 3 neurons (equal to the number of classes)
1. Two-layer network with small/medium/large hidden layer
1. Using more layers

Make sure you observe overfitting when you are using rich model with lots of neurons (parameters).

In [ ]:
# Experiment

## Part 2: MNIST Training

Both Keras and PyTorch contain MNIST as built-in dataset, so you can easily get it with a couple of lines of code ([Keras](https://keras.io/api/datasets/mnist/), [PyTorch](https://pytorch.org/vision/stable/datasets.html)). You will also be able to load both train and test datasets without manually splitting them.

In [ ]:
# Load the dataset

Now you need to perform the steps above to make sure dataset is normalized (it would probably already be), defining and training a neural network.

## Takeaway

1. Neural networks can be used for traditional machine learning tasks. However, they are in many cases too powerful, and can cause overfitting.
1. It is important in this assignment that you observe the overfitting behavior, and try to avoid it.
1. With frameworks like Keras, sometimes training a neural network is quite straightforward. But you need to understand what goes on.